# 데이터 스토어를 사용하는 Amazon Bedrock AgentCore Gateway 인터셉터 기반 세분화된 액세스 제어

## 개요

이 노트북에서는 권한 관리를 위한 데이터 스토어로 **Amazon DynamoDB**를 사용하고 **Gateway 인터셉터**를 통해 **Amazon Bedrock AgentCore Gateway**에 **세분화된 액세스 제어(FGAC)**를 적용하는 방법을 보여 줍니다. 재배포가 필요한 정책 기반 접근 방식과 달리, 이 패턴은 DynamoDB의 레코드를 업데이트하여 **도구 권한을 동적으로 관리**할 수 있습니다.

### 액세스 제어에 데이터 스토어를 사용하는 이유

기존 접근 방식은 OAuth 범위 또는 정적 정책에 권한을 포함합니다. 이 튜토리얼에서는 다음과 같은 더 유연한 접근 방식을 소개합니다.

- **동적 권한 관리**: 인프라를 재배포하지 않고 DynamoDB에서 도구 액세스 권한 업데이트
- **중앙 집중식 정책 스토어**: 모든 클라이언트 권한을 한곳에서 관리
- **확장성**: 여러 클라이언트와 도구에 걸친 복잡한 권한 매트릭스 처리
- **감사 가능성**: DynamoDB의 기본 제공 기능을 통해 권한 변경 사항 추적

Gateway 인터셉터는 런타임에 Amazon DynamoDB 테이블을 쿼리하여 각 클라이언트가 액세스할 수 있는 도구를 결정함으로써 실시간 액세스 제어를 제공합니다.

---

## 이 튜토리얼에서 다루는 내용

이 튜토리얼에서는 RESPONSE 인터셉터를 사용하여 **도구 목록 조회(List Tools)** 작업에 FGAC를 구현합니다.

📋 **FGAC가 적용된 도구 목록 조회(RESPONSE 인터셉터)**  
   - Gateway의 tools/list 응답을 가로챕니다.
   - Amazon DynamoDB를 쿼리하여 클라이언트에 허용된 도구를 가져옵니다(JWT의 Client ID로 식별).
   - 권한이 부여된 도구만 표시하도록 도구 목록을 필터링합니다.
   - 필터링된 응답을 클라이언트에 반환합니다.

FGAC가 적용된 **도구 호출(Invoke Tool)** 작업은 다음 튜토리얼을 참조하세요. [사용자 지정 범위를 사용하는 세분화된 액세스 제어](../01-fine-grained-access-control-using-custom-scopes.ipynb)

![도구 목록](../images/FGAC_data_store.png)

---

## Gateway 인터셉터를 사용하는 이유

Gateway 인터셉터를 사용하면 다음 작업을 수행할 수 있습니다.

- **세분화된 액세스 제어 구현**: 클라이언트별, 도구별 권한 부여 규칙 적용
- **사용자 지정 권한 부여 로직 삽입**: 동적 권한을 위해 외부 데이터 스토어 쿼리
- **감사 및 거버넌스**: 규정 준수를 위해 도구 액세스 시도 기록
- **요청/응답 변환**: 전송 중인 데이터 필터링, 마스킹 또는 수정

인터셉터는 **Gateway 계층**에 연결되므로 애플리케이션 코드를 수정하지 않고도 하위의 **모든** MCP 서버 또는 런타임에 중앙 집중식 정책을 적용합니다.

---

## 튜토리얼 세부 정보

| 정보                     | 세부 정보                                                                                      |
|--------------------------|-------------------------------------------------------------------------------------------------|
| **튜토리얼 유형**        | 인터랙티브                                                                                     |
| **AgentCore 구성 요소**  | Amazon Bedrock AgentCore Gateway, Gateway 인터셉터                                             |
| **Gateway 대상 유형**    | MCP 서버(AgentCore Runtime에서 실행되는 FastMCP)                                              |
| **인터셉터 유형**        | AWS Lambda(RESPONSE)                                                                           |
| **인바운드 인증 IdP**    | Amazon Cognito(CUSTOM_JWT 권한 부여자)                                                        |
| **데이터 스토어**        | Amazon DynamoDB(클라이언트-도구 권한 매핑 저장)                                               |
| **액세스 제어**          | JWT의 Client ID와 DynamoDB 권한 조회를 사용하는 FGAC                                          |
| **튜토리얼 구성 요소**   | Gateway, Runtime MCP 서버, Amazon Cognito, Gateway 인터셉터, MCP 도구, Amazon DynamoDB        |
| **튜토리얼 분야**        | 범용                                                                                            |
| **예제 난이도**          | 중급                                                                                            |
| **사용 SDK**             | boto3                                                                                           |

---

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.

- Jupyter notebook(Python 커널)
- 다음 서비스에 대한 권한이 있는 AWS 자격 증명
  - AWS Lambda
  - AWS IAM
  - Amazon Cognito
  - Amazon DynamoDB
  - Amazon Bedrock AgentCore 서비스(컨트롤 플레인 및 런타임)
- Python 3.9 이상
- AWS Lambda, IAM 역할, Amazon Cognito 및 Amazon Bedrock AgentCore Gateway에 대한 기본 지식

> ⚠️ **참고:** 마지막의 정리 섹션에서는 이 튜토리얼에서 생성한 AWS 리소스(Gateway, Lambda, IAM 역할 등)를 삭제합니다. 모든 항목을 삭제할 준비가 되었을 때만 실행하세요.


In [ ]:
# 필수 라이브러리 가져오기
import boto3
import time
import sys
import requests
from pathlib import Path
from datetime import datetime

# 경로에 utils 추가
current_dir = Path.cwd()
utils_dir = current_dir.parent.parent
sys.path.insert(0, str(utils_dir))

import utils

print("✓ Libraries imported")

# 이 배포의 고유 식별자 생성
DEPLOYMENT_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
print(f"\nDeployment ID: {DEPLOYMENT_ID}")

# 구성
REGION = "us-east-1"

# 리소스 이름
USER_POOL_NAME = f"gateway-pool-{DEPLOYMENT_ID}"
DYNAMODB_TABLE_NAME = f"ClientToolPermissions-{DEPLOYMENT_ID}"
LAMBDA_FUNCTION_NAME = f"interceptor-lambda-{DEPLOYMENT_ID}"
LAMBDA_ROLE_NAME = f"interceptor-lambda-role-{DEPLOYMENT_ID}"
GATEWAY_NAME = f"interceptor-gateway-{DEPLOYMENT_ID}"

print("Configuration:")
print(f"  Region: {REGION}")
print(f"  User Pool: {USER_POOL_NAME}")
print(f"  DynamoDB Table: {DYNAMODB_TABLE_NAME}")
print(f"  Lambda Function: {LAMBDA_FUNCTION_NAME}")
print(f"  Lambda Role: {LAMBDA_ROLE_NAME}")
print(f"  Gateway Name: {GATEWAY_NAME}")

# AWS 클라이언트 초기화
cognito_client = boto3.client("cognito-idp", region_name=REGION)
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
print("\n✓ AWS clients initialized")

---

## 파트 1: 설정 및 배포

### 1.1단계: Amazon Cognito 사용자 풀 및 앱 클라이언트 생성

서로 다른 애플리케이션 또는 서비스를 나타내는 여러 앱 클라이언트를 생성합니다.

#### Client ID와 액세스 제어

**Client ID**는 Amazon Cognito에 등록된 각 애플리케이션의 고유 식별자입니다. 클라이언트가 인증하면 JWT 토큰에 Client ID가 포함되며, Gateway 인터셉터는 이를 사용하여 DynamoDB에서 권한을 조회합니다.

**핵심 사항:**
- 모든 클라이언트는 Gateway 액세스에 동일한 OAuth 범위(`gateway/tools`)를 사용합니다.
- **도구 수준 권한**은 DynamoDB에 저장되며 Client ID를 사용하여 쿼리됩니다.
- OAuth를 재구성하지 않고도 권한을 동적으로 업데이트할 수 있습니다.

**보안 모범 사례:**

⚠️ 신뢰할 수 있는 ID 공급자(Amazon Cognito, Okta, Auth0, Azure AD 등)가 **암호학적으로 서명한 JWT 토큰을 항상 사용하세요**. Gateway는 요청을 처리하기 전에 서명을 검증하여 Client ID가 신뢰할 수 있고 변조되지 않았음을 보장합니다.

**피해야 할 사항:**
- ❌ 인증에 사용자 지정 헤더(예: `X-Client-ID`)를 사용하지 마세요. 쉽게 위조될 수 있습니다.
- ❌ 인증 결정에 사용할 Client ID를 쿼리 파라미터로 전달하지 마세요.
- ❌ 서명되지 않았거나 검증되지 않은 토큰을 사용하지 마세요.


Gateway는 인터셉터에 요청을 전달하기 전에 JWT 서명을 검증하여 토큰의 Client ID가 인증되었음을 보장합니다.

**고급:** 멀티 에이전트 시나리오에서는 더 세분화된 제어를 위해 Client ID와 Agent ID를 `{ClientID}#{AgentID}` 형식으로 조합합니다.


In [ ]:
# 여러 앱 클라이언트가 있는 Cognito 사용자 풀 생성
# 사용자 풀 생성 또는 가져오기
USER_POOL_ID = utils.get_or_create_user_pool(cognito_client, USER_POOL_NAME)

# 리소스 서버 생성 또는 가져오기
RESOURCE_SERVER_ID = "gateway"
RESOURCE_SERVER_NAME = "Gateway Resource Server"
SCOPES = [{"ScopeName": "tools", "ScopeDescription": "Access to gateway tools"}]
utils.get_or_create_resource_server(cognito_client, USER_POOL_ID, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)

# 리소스 서버 전파 대기
time.sleep(3)

# 서로 다른 권한 수준에 사용할 여러 앱 클라이언트 생성
clients = {}
for client_name in ["full-access", "readonly", "calculator", "data"]:
    client_id, client_secret = utils.get_or_create_m2m_client(
        cognito_client,
        USER_POOL_ID,
        f"{client_name}-client-{DEPLOYMENT_ID}",
        RESOURCE_SERVER_ID,
        ["gateway/tools"],
    )
    clients[client_name] = {"client_id": client_id, "client_secret": client_secret}
    print(f"✓ Created/found client: {client_name}")

# 간편하게 액세스할 수 있도록 클라이언트 ID 추출
CLIENT_ID_FULL = clients["full-access"]["client_id"]
CLIENT_ID_READONLY = clients["readonly"]["client_id"]
CLIENT_ID_CALCULATOR = clients["calculator"]["client_id"]
CLIENT_ID_DATA = clients["data"]["client_id"]

# OAuth URL 구성
POOL_DOMAIN = USER_POOL_ID.replace("_", "").lower()
DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
TOKEN_URL = f"https://{POOL_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"

print("\n✓ Cognito setup complete")
print(f"  User Pool ID: {USER_POOL_ID}")
print(f"  Discovery URL: {DISCOVERY_URL}")
print(f"  Token URL: {TOKEN_URL}")

### 1.2단계: Amazon DynamoDB 권한 테이블 생성

클라이언트와 도구 간의 권한 매핑을 저장할 테이블을 생성합니다. 각 레코드는 하나의 클라이언트에 하나의 도구에 대한 액세스 권한을 부여합니다.
- **ClientID**(파티션 키): Amazon Cognito Client ID
- **ToolName**(정렬 키): 도구 이름
- **Allowed**: 부울 플래그

예: `ClientID: abc123, ToolName: weather_tool, Allowed: True`

이 모델은 각 Client ID가 하나의 애플리케이션을 나타낸다고 가정합니다. 사용자 수준 권한에는 `{ClientID}#{UserID}`와 같은 복합 키를 사용합니다. Lambda 인터셉터는 JWT의 Client ID를 사용하여 이 테이블을 쿼리합니다.

In [ ]:
# DynamoDB 테이블 생성
utils.create_dynamodb_table(
    table_name=DYNAMODB_TABLE_NAME,
    key_schema=[
        {"AttributeName": "ClientID", "KeyType": "HASH"},
        {"AttributeName": "ToolName", "KeyType": "RANGE"},
    ],
    attribute_definitions=[
        {"AttributeName": "ClientID", "AttributeType": "S"},
        {"AttributeName": "ToolName", "AttributeType": "S"},
    ],
    region=REGION,
)

### 1.3단계: DynamoDB에 클라이언트 권한 로드
각 Cognito client_id를 허용된 도구에 매핑합니다.

In [ ]:
# 클라이언트 권한 매핑 정의(간소화된 형식)
CLIENT_PERMISSIONS = {
    CLIENT_ID_FULL: [
        "weather_tool",
        "database_query_tool",
        "calculation_tool",
        "search_tool",
        "file_handler_tool",
    ],
    CLIENT_ID_READONLY: ["weather_tool", "search_tool"],
    CLIENT_ID_CALCULATOR: ["calculation_tool"],
    CLIENT_ID_DATA: ["database_query_tool", "file_handler_tool", "calculation_tool"],
}

# 매핑에서 권한 목록 생성
SAMPLE_PERMISSIONS = [
    {"ClientID": client_id, "ToolName": tool_name, "Allowed": True}
    for client_id, tools in CLIENT_PERMISSIONS.items()
    for tool_name in tools
]

# DynamoDB에 권한 로드
utils.batch_write_dynamodb(table_name=DYNAMODB_TABLE_NAME, items=SAMPLE_PERMISSIONS, region=REGION)

### 1.4단계: Lambda 인터셉터용 IAM 역할 생성
Lambda에 DynamoDB 읽기 및 CloudWatch 로그 쓰기 권한을 부여합니다.

In [ ]:
# Lambda 인터셉터용 IAM 역할 생성
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]
table_arn = f"arn:aws:dynamodb:{REGION}:{account_id}:table/{DYNAMODB_TABLE_NAME}"

dynamodb_policy = {
    "Effect": "Allow",
    "Action": ["dynamodb:Query", "dynamodb:GetItem"],
    "Resource": table_arn,
}

LAMBDA_ROLE_ARN = utils.create_lambda_role_with_policies(
    role_name=LAMBDA_ROLE_NAME,
    policy_statements=[dynamodb_policy],
    description="Lambda interceptor role with DynamoDB access",
)

print(f"✓ Lambda role ready: {LAMBDA_ROLE_ARN}")

### 1.5단계: Lambda 인터셉터 함수 배포
Lambda는 JWT에서 client_id를 추출하고 DynamoDB 권한을 기준으로 도구를 필터링합니다.

In [ ]:
# utils를 사용하여 Lambda 인터셉터 배포
LAMBDA_ARN = utils.deploy_lambda_function(
    function_name=LAMBDA_FUNCTION_NAME,
    role_arn=LAMBDA_ROLE_ARN,
    lambda_code_path="src/lambda/lambda_function.py",
    environment_vars={
        "PERMISSIONS_TABLE_NAME": DYNAMODB_TABLE_NAME,
        "DYNAMODB_REGION": REGION,
    },
    region=REGION,
)

# Gateway에 Lambda 호출 권한 부여
utils.grant_gateway_invoke_permission(function_name=LAMBDA_FUNCTION_NAME, region=REGION)

print(f"\n✓ Lambda interceptor deployed and configured: {LAMBDA_ARN}")

### 1.6단계: 응답 인터셉터가 있는 Gateway 생성

**RESPONSE 인터셉터를 사용하는 이유는 무엇인가요?**  
Gateway가 모든 대상에서 도구를 수집한 후 집계된 도구 목록을 필터링합니다. 인터셉터는 JWT의 Client ID를 사용하여 DynamoDB를 쿼리한 다음 허용된 도구만 반환합니다.

**흐름:** 요청 → JWT 검증 → 도구 집계 → DynamoDB 조회 → 필터링된 응답


In [ ]:
# Gateway IAM 역할 생성
gateway_iam_role = utils.create_agentcore_gateway_role_with_region(GATEWAY_NAME, REGION)
GATEWAY_ROLE_ARN = gateway_iam_role["Role"]["Arn"]

print(f"✓ Gateway role created: {GATEWAY_ROLE_ARN}")

# 역할 전파 대기
time.sleep(10)

# Lambda 인터셉터가 있는 Gateway 생성
print("\nCreating Gateway with RESPONSE interceptor...")

try:
    gateway_response = gateway_client.create_gateway(
        name=GATEWAY_NAME,
        protocolType="MCP",
        protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26"]}},
        interceptorConfigurations=[
            {
                "interceptor": {"lambda": {"arn": LAMBDA_ARN}},
                "interceptionPoints": ["RESPONSE"],
                "inputConfiguration": {"passRequestHeaders": True},
            }
        ],
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": DISCOVERY_URL,
                "allowedClients": [
                    CLIENT_ID_FULL,
                    CLIENT_ID_DATA,
                    CLIENT_ID_CALCULATOR,
                    CLIENT_ID_READONLY,
                ],
            }
        },
        roleArn=GATEWAY_ROLE_ARN,
    )

    GATEWAY_ID = gateway_response.get("gatewayId")
    print(f"✓ Gateway created: {GATEWAY_ID}")

except Exception as e:
    print(f"\n✗ Failed to create Gateway: {e}")
    raise

In [ ]:
# 서명된 요청을 사용하여 Gateway가 준비될 때까지 대기
print("\nWaiting for Gateway to be ready...")

max_attempts = 30
for attempt in range(max_attempts):
    try:
        response = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
        status = response.get("status", "UNKNOWN")

        print(f"  [{attempt + 1}/{max_attempts}] Status: {status}")

        if status == "READY":
            GATEWAY_URL = response.get("gatewayUrl")
            print("\n✓ Gateway is ready!")
            print(f"  URL: {GATEWAY_URL}")
            print(f"  Interceptor: RESPONSE (Lambda: {LAMBDA_ARN})")
            break
        elif status == "FAILED":
            print("\n✗ Gateway creation failed")
            raise Exception("Gateway failed")
    except Exception as e:
        print(f"  [{attempt + 1}/{max_attempts}] Error: {e}")
        raise

    time.sleep(10)
else:
    print("\n⚠ Timeout waiting for Gateway")
    raise Exception("Gateway timeout")

### 1.7단계: Gateway에 샘플 도구 등록
도구 Lambda를 배포하고 Gateway 대상으로 등록합니다.

In [ ]:
# 도구 모듈 가져오기

sys.path.insert(0, str(Path.cwd()))

from src.tools import (
    weather_tool,
    database_query_tool,
    calculation_tool,
    search_tool,
    file_handler_tool,
)

# 도구 Lambda용 IAM 역할 생성
TOOL_ROLE_ARN = utils.create_lambda_role(
    role_name=f"tool-lambda-role-{DEPLOYMENT_ID}",
    description="Role for tool Lambda functions",
)

# 도구 Lambda 함수 가져오기 및 배포
print("Deploying tool Lambda functions...")
sys.path.insert(0, str(Path.cwd()))

lambda_client = boto3.client("lambda", region_name=REGION)

tools_to_deploy = [
    ("weather_tool", weather_tool),
    ("database_query_tool", database_query_tool),
    ("calculation_tool", calculation_tool),
    ("search_tool", search_tool),
    ("file_handler_tool", file_handler_tool),
]

deployed_tools = []

for tool_name, tool_module in tools_to_deploy:
    print(f"  Deploying {tool_name}...")

    function_name = f"{tool_name.replace('_', '-')}-{DEPLOYMENT_ID}"
    tool_code_path = Path(tool_module.__file__)

    lambda_arn = utils.deploy_lambda_function(
        function_name=function_name,
        role_arn=TOOL_ROLE_ARN,
        lambda_code_path=str(tool_code_path),
        environment_vars={"TOOL_NAME": tool_name},
        description=f"{tool_name} function",
        region=REGION,
    )

    tool_definition = getattr(
        tool_module,
        "TOOL_DEFINITION",
        {"name": tool_name, "description": f"{tool_name} function"},
    )

    deployed_tools.append(
        {
            "tool_name": tool_name,
            "function_name": function_name,
            "lambda_arn": lambda_arn,
            "tool_definition": tool_definition,
        }
    )

print(f"✓ Deployed {len(deployed_tools)} tool Lambdas")

In [ ]:
# 도구를 Gateway 대상으로 등록
print("Registering tools as Gateway targets...")
created_targets = []

for tool in deployed_tools:
    print(f"  Registering {tool['tool_name']}...")

    try:
        response = gateway_client.create_gateway_target(
            gatewayIdentifier=GATEWAY_ID,
            name=f"{tool['tool_name'].replace('_', '-')}-target",
            targetConfiguration={
                "mcp": {
                    "lambda": {
                        "lambdaArn": tool["lambda_arn"],
                        "toolSchema": {"inlinePayload": [tool["tool_definition"]]},
                    }
                }
            },
            credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
        )

        target_id = response["targetId"]
        print(f"    ✓ Target created: {target_id}")

        # 대상이 READY 상태가 될 때까지 대기
        for attempt in range(18):
            status_response = gateway_client.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=target_id)
            status = status_response.get("status")

            if status == "READY":
                print("    ✓ Target is READY")
                created_targets.append(
                    {
                        "tool_name": tool["tool_name"],
                        "target_id": target_id,
                        "lambda_arn": tool["lambda_arn"],
                    }
                )
                break
            elif status == "FAILED":
                print("    ✗ Target FAILED")
                break

            time.sleep(10)

    except Exception as e:
        print(f"    ✗ Failed: {e}")

print(f"✓ Registered {len(created_targets)}/{len(deployed_tools)} targets")

# 정리를 위해 저장
DEPLOYED_TOOL_FUNCTIONS = [t["function_name"] for t in deployed_tools]
CREATED_TARGET_IDS = [t["target_id"] for t in created_targets]

---

## 파트 2: 테스트

### 2.1단계: 서로 다른 Client ID로 테스트
각 클라이언트에 허용된 도구만 표시되는지 확인합니다.

In [ ]:
# 예상 권한과 함께 테스트 클라이언트 정의
print("=" * 80)
print("Testing Fine-Grained Access Control with Different Clients")
print("=" * 80)

test_clients = [
    {
        "name": "full-access",
        "client_id": CLIENT_ID_FULL,
        "expected_tools": [
            "weather_tool",
            "database_query_tool",
            "calculation_tool",
            "search_tool",
            "file_handler_tool",
        ],
    },
    {
        "name": "readonly",
        "client_id": CLIENT_ID_READONLY,
        "expected_tools": ["weather_tool", "search_tool"],
    },
    {
        "name": "calculator",
        "client_id": CLIENT_ID_CALCULATOR,
        "expected_tools": ["calculation_tool"],
    },
    {
        "name": "data",
        "client_id": CLIENT_ID_DATA,
        "expected_tools": [
            "database_query_tool",
            "file_handler_tool",
            "calculation_tool",
        ],
    },
]

print(f"\n✓ Configured {len(test_clients)} test clients")

In [ ]:
# utils를 사용하여 Cognito에서 클라이언트 보안 암호 가져오기
client_secrets = utils.get_client_secrets(
    cognito_client=cognito_client,
    user_pool_id=USER_POOL_ID,
    client_configs=test_clients,
)

In [ ]:
# 각 클라이언트의 도구 액세스 테스트
test_results = []

for client_config in test_clients:
    print(f"\n{'=' * 60}")
    print(f"Testing Client: {client_config['name']}")
    print(f"{'=' * 60}")
    print(f"  Client ID: {client_config['client_id']}")
    print(f"  Expected tools: {client_config['expected_tools']}")

    client_id = client_config["client_id"]
    client_secret = client_secrets.get(client_id)

    if not client_secret:
        print("  ✗ No client secret available, skipping")
        test_results.append({"name": client_config["name"], "passed": False, "reason": "No secret"})
        continue

    try:
        # 1단계: TOKEN_URL을 사용하여 액세스 토큰 가져오기
        print("\n  Step 1: Requesting access token...")
        print(f"  Token URL: {TOKEN_URL}")
        time.sleep(2)  # 잠시 대기

        token_data = utils.get_token(
            user_pool_id=USER_POOL_ID,
            client_id=client_id,
            client_secret=client_secret,
            scope_string="gateway/tools",
            REGION=REGION,
        )

        if "error" in token_data:
            print(f"    ✗ Token request failed: {token_data['error']}")
            test_results.append(
                {
                    "name": client_config["name"],
                    "passed": False,
                    "reason": "Token failed",
                }
            )
            continue

        token = token_data["access_token"]
        print(f"    ✓ Token obtained (expires in {token_data.get('expires_in')}s)")

        # 2단계: MCP 프로토콜을 사용하여 Gateway의 도구 목록 호출
        print("\n  Step 2: Calling Gateway to list tools...")

        # MCP tools/list 요청
        mcp_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}

        response = requests.post(
            GATEWAY_URL,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json",
            },
            json=mcp_request,
        )

        if response.status_code != 200:
            print(f"    ✗ Gateway request failed: {response.status_code}")
            print(f"    Response: {response.text}")
            test_results.append(
                {
                    "name": client_config["name"],
                    "passed": False,
                    "reason": f"HTTP {response.status_code}",
                }
            )
            continue

        result = response.json()

        if "error" in result:
            print(f"    ✗ MCP error: {result['error']}")
            test_results.append({"name": client_config["name"], "passed": False, "reason": "MCP error"})
            continue

        # 도구 이름 추출
        tools = result.get("result", {}).get("tools", [])
        actual_tool_names = [tool["name"].split("___")[1] if "___" in tool["name"] else tool["name"] for tool in tools]

        print(f"    ✓ Received {len(actual_tool_names)} tools")
        print(f"    Parsed names: {actual_tool_names}")

        # 3단계: 권한 확인
        print("\n  Step 3: Verifying permissions...")

        expected_tools = set(client_config["expected_tools"])
        actual_tools = set(actual_tool_names)

        print(f"    Expected: {sorted(expected_tools)}")
        print(f"    Actual:   {sorted(actual_tools)}")

        if expected_tools == actual_tools:
            print("\n  ✅ PASS: Client has correct permissions")
            test_results.append({"name": client_config["name"], "passed": True})
        else:
            print("\n  ❌ FAIL: Permission mismatch")

            missing = expected_tools - actual_tools
            if missing:
                print(f"    Missing tools: {sorted(missing)}")

            extra = actual_tools - expected_tools
            if extra:
                print(f"    Extra tools: {sorted(extra)}")

            test_results.append({"name": client_config["name"], "passed": False, "reason": "Mismatch"})

    except Exception as e:
        print(f"\n  ✗ Test failed with exception: {e}")
        import traceback

        traceback.print_exc()
        test_results.append({"name": client_config["name"], "passed": False, "reason": str(e)})

In [ ]:
# 테스트 요약 표시
print(f"\n{'=' * 80}")
print("Test Summary")
print(f"{'=' * 80}")

passed_count = sum(1 for r in test_results if r["passed"])
total_count = len(test_results)

for result in test_results:
    status = "✅ PASS" if result["passed"] else "❌ FAIL"
    reason = f" ({result.get('reason', '')})" if not result["passed"] and "reason" in result else ""
    print(f"  {status}: {result['name']}{reason}")

print(f"\nTotal: {passed_count}/{total_count} passed")

if passed_count == total_count:
    print("\n🎉 All tests passed! Fine-grained access control is working correctly.")
else:
    print("\n⚠️  Some tests failed. Check the logs above for details.")

---

## 파트 3: 정리

⚠️ **경고: 파트 1에서 생성한 모든 리소스가 삭제됩니다!**

모든 항목을 정리하려는 경우에만 이 섹션을 실행하세요.

### 3.1단계: 생성된 리소스 삭제

In [ ]:
# 정리 - 생성된 모든 리소스 삭제
print("Starting cleanup...")

# 1. Gateway 대상 삭제
if "CREATED_TARGET_IDS" in globals() and "GATEWAY_ID" in globals():
    utils.delete_gateway_targets(gateway_client, GATEWAY_ID, CREATED_TARGET_IDS)

# 2. Gateway 삭제
if "GATEWAY_ID" in globals():
    utils.delete_gateway(gateway_client, GATEWAY_ID)
    print("✓ Deleted gateway")

# 3. Lambda 함수 삭제(도구 및 인터셉터)
lambda_functions_to_delete = []
if "DEPLOYED_TOOL_FUNCTIONS" in globals():
    lambda_functions_to_delete.extend(DEPLOYED_TOOL_FUNCTIONS)
if "LAMBDA_FUNCTION_NAME" in globals():
    lambda_functions_to_delete.append(LAMBDA_FUNCTION_NAME)

if lambda_functions_to_delete:
    utils.delete_lambda_functions(lambda_functions_to_delete, REGION)

# 4. IAM 역할 삭제
if "LAMBDA_ROLE_NAME" in globals():
    utils.delete_iam_role(LAMBDA_ROLE_NAME)
if "TOOL_ROLE_NAME" in globals():
    utils.delete_iam_role(f"tool-lambda-role-{DEPLOYMENT_ID}")
if "GATEWAY_ROLE_ARN" in globals():
    utils.delete_iam_role(f"gateway-role-{DEPLOYMENT_ID}")

# 5. DynamoDB 테이블 삭제
if "DYNAMODB_TABLE_NAME" in globals():
    utils.delete_dynamodb_table(DYNAMODB_TABLE_NAME, REGION)

# 6. Cognito 사용자 풀 삭제
if "USER_POOL_ID" in globals():
    utils.delete_cognito_user_pool(USER_POOL_ID, REGION)

print("\n✓ Cleanup complete!")

---

# 요약

이 노트북에서는 전체 수명 주기를 완료했습니다.

1. ✅ **설정** - DynamoDB, Lambda, IAM 역할 및 Gateway 생성
2. ✅ **테스트** - 실제 Gateway를 통한 도구 필터링 확인
3. ✅ **정리** - 모든 리소스 삭제

## 구현한 내용

- DynamoDB 권한을 사용하는 **에이전트 기반 도구 필터링**
- Gateway 응답을 수정하는 **Lambda RESPONSE 인터셉터**
- 요청 체인을 통한 **사용자 지정 헤더 전파**(Agent-ID)
- **전체 리소스 수명 주기** 관리

## 다음 단계

- 다른 구성으로 다시 실행
- 사용자 지정 에이전트와 도구 추가
- 실제 AgentCore Runtime 에이전트와 통합
- 디버깅을 위해 CloudWatch 로그 모니터링